<a href="https://colab.research.google.com/github/ekee7/acc_reality_check/blob/main/ACC_Reality_Check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ACC Reality Check**

**Group Members: Emma Kee (hvh6ng), Kylie Sharp (dhq4mr), and Raffaella Alpaca (bet3at)**

In [1]:
from pathlib import Path

# Create folders for raw data
Path("data/raw/torvik").mkdir(parents=True, exist_ok=True)
Path("data/raw/scoreboard").mkdir(parents=True, exist_ok=True)
Path("data/raw/boxscores").mkdir(parents=True, exist_ok=True)

print("Project folders created")

Project folders created


## **Import Data**

In [2]:
import requests
import pandas as pd

# Download Torvik data
url = "https://raw.githubusercontent.com/ekee7/acc_reality_check/main/2027_team_results.csv"

response = requests.get(url)
response.raise_for_status()

# Save original CSV
file_path = Path("data/raw/torvik/2027_team_results.csv")
file_path.write_bytes(response.content)

# Load CSV into pandas
df_torvik = pd.read_csv(file_path)

# Preserve original copy
df_torvik_original = df_torvik.copy()

print("Dataset downloaded successfully")
print("Dataset shape:", df_torvik.shape)

display(df_torvik.head())

Dataset downloaded successfully
Dataset shape: (365, 45)


,rank,team,conf,record,adjoe,oe Rank,adjde,de Rank,barthag,rank.1,...,ConPA,ConPoss,ConOE,ConDE,ConSOSRemain,Conf Win%,WAB,WAB Rk,Fun Rk,adjt
0,1,Houston,B12,6-0,124.474683,3,93.804798,3,0.962788,1,...,0.0,0.0,0,0,0.831863,0,2.444834,1,26,67.558509
1,2,Illinois,B10,5-1,126.281293,1,96.602043,16,0.956102,2,...,0.0,0.0,0,0,0.836524,0,0.985492,20,120,73.530920
2,3,Kansas,B12,5-1,122.691516,6,94.294347,4,0.953797,3,...,0.0,0.0,0,0,0.829842,0,1.296530,9,161,72.710877
3,4,Florida,SEC,6-1,124.762015,2,96.285008,13,0.951644,4,...,0.0,0.0,0,0,0.856907,0,2.286053,2,68,68.495192
4,5,Duke,ACC,4-2,119.887207,20,93.240421,1,0.947388,5,...,0.0,0.0,0,0,0.801220,0,-0.309751,88,336,71.564028


In [14]:
# Overview of df_torvik
print("Number of rows:", len(df_torvik))
print("Number of columns:", len(df_torvik.columns))

print("\nColumn names:")
print(df_torvik.columns.tolist())

Number of rows: 365
Number of columns: 45

Column names:
['rank', 'team', 'conf', 'record', 'adjoe', 'oe Rank', 'adjde', 'de Rank', 'barthag', 'rank.1', 'proj. W', 'Proj. L', 'Pro Con W', 'Pro Con L', 'Con Rec.', 'sos', 'ncsos', 'consos', 'Proj. SOS', 'Proj. Noncon SOS', 'Proj. Con SOS', 'elite SOS', 'elite noncon SOS', 'Opp OE', 'Opp DE', 'Opp Proj. OE', 'Opp Proj DE', 'Con Adj OE', 'Con Adj DE', 'Qual O', 'Qual D', 'Qual Barthag', 'Qual Games', 'FUN', 'ConPF', 'ConPA', 'ConPoss', 'ConOE', 'ConDE', 'ConSOSRemain', 'Conf Win%', 'WAB', 'WAB Rk', 'Fun Rk', 'adjt']


In [16]:
# Filter the dataset to ACC teams
df_acc = df_torvik[df_torvik["conf"] == "ACC"].copy()

# Check the results
print("Number of ACC teams:", len(df_acc))
print("\nACC teams:")
display(df_acc[["team", "conf", "record"]].sort_values("team"))

Number of ACC teams: 18

ACC teams:


,team,conf,record
100,Boston College,ACC,6-0
99,California,ACC,4-2
67,Clemson,ACC,4-1
4,Duke,ACC,4-2
54,Florida St.,ACC,5-1
122,Georgia Tech,ACC,5-1
23,Louisville,ACC,3-1
14,Miami FL,ACC,5-1
32,N.C. State,ACC,5-1
51,North Carolina,ACC,4-1


In [5]:
# Save the ACC teams to a separate CSV file
acc_file_path = Path("data/raw/torvik/acc_teams.csv")
df_acc.to_csv(acc_file_path, index=False)

print("ACC dataset saved successfully")
print("Number of teams saved:", len(df_acc))

ACC dataset saved successfully
Number of teams saved: 18


In [7]:
from datetime import date, timedelta
import json

# ESPN scoreboard API
scoreboard_url = ("https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard")

# Set the date range
start_date = date(2026, 11, 1)
end_date = date(2026, 11, 23)

# Store all scoreboard responses
all_scoreboards = {}

current_date = start_date

while current_date <= end_date:
    date_string = current_date.strftime("%Y%m%d")

    params = {
        "groups": 2,
        "dates": date_string,
        "limit": 200
    }

    response = requests.get(scoreboard_url, params=params)
    response.raise_for_status()

    scoreboard_data = response.json()

    # Save the original ESPN response
    file_path = Path(f"data/raw/scoreboard/scoreboard_{date_string}.json")

    file_path.write_text(json.dumps(scoreboard_data, indent=2))

    all_scoreboards[date_string] = scoreboard_data

    print(
        date_string,
        "- Games found:",
        len(scoreboard_data.get("events", []))
    )

    current_date += timedelta(days=1)

print("\nScoreboard data collection complete")

20261101 - Games found: 1
20261102 - Games found: 13
20261103 - Games found: 4
20261104 - Games found: 1
20261105 - Games found: 7
20261106 - Games found: 9
20261107 - Games found: 1
20261108 - Games found: 3
20261109 - Games found: 5
20261110 - Games found: 10
20261111 - Games found: 2
20261112 - Games found: 0
20261113 - Games found: 11
20261114 - Games found: 3
20261115 - Games found: 3
20261116 - Games found: 2
20261117 - Games found: 10
20261118 - Games found: 3
20261119 - Games found: 5
20261120 - Games found: 9
20261121 - Games found: 3
20261122 - Games found: 2
20261123 - Games found: 3

Scoreboard data collection complete


In [18]:
# Combine games from all saved scoreboard responses
all_games = []

for date_string, scoreboard in all_scoreboards.items():
    for game in scoreboard.get("events", []):
        all_games.append(game)

# Create a DataFrame with basic game information
games_summary = []

for game in all_games:
    competition = game["competitions"][0]

    teams = competition["competitors"]

    games_summary.append({
        "game_id": game["id"],
        "game_name": game["name"],
        "game_date_utc": game["date"],
        "status": competition["status"]["type"]["name"],
        "home_team": next(
            (team["team"]["displayName"] for team in teams
             if team["homeAway"] == "home"), None
        ),
        "away_team": next(
            (team["team"]["displayName"] for team in teams
             if team["homeAway"] == "away"), None
        )
    })

df_games_summary = pd.DataFrame(games_summary)

print("Total games collected:", len(df_games_summary))
print("\nGame statuses:")
print(df_games_summary["status"].value_counts())

display(df_games_summary.head(10))

Total games collected: 110

Game statuses:
status
STATUS_FINAL        108
STATUS_SCHEDULED      2
Name: count, dtype: int64


,game_id,game_name,game_date_utc,status,home_team,away_team
0,401920982,Notre Dame Fighting Irish at Villanova Wildcats,2026-11-01T14:30Z,STATUS_FINAL,Villanova Wildcats,Notre Dame Fighting Irish
1,401906769,Miami Hurricanes at Florida Gators,2026-11-02T23:00Z,STATUS_FINAL,Florida Gators,Miami Hurricanes
2,401909562,Army Black Knights at Duke Blue Devils,2026-11-02T23:30Z,STATUS_FINAL,Duke Blue Devils,Army Black Knights
3,401918947,Chattanooga Mocs at Louisville Cardinals,2026-11-03T02:00Z,STATUS_FINAL,Louisville Cardinals,Chattanooga Mocs
4,401918073,Stetson Hatters at Clemson Tigers,2026-11-03T00:00Z,STATUS_FINAL,Clemson Tigers,Stetson Hatters
5,401913099,Western Carolina Catamounts at North Carolina ...,2026-11-03T00:00Z,STATUS_FINAL,North Carolina Tar Heels,Western Carolina Catamounts
6,401909716,New Haven Chargers at Syracuse Orange,2026-11-03T00:00Z,STATUS_FINAL,Syracuse Orange,New Haven Chargers
7,401909705,Rider Broncs at Pittsburgh Panthers,2026-11-03T00:00Z,STATUS_FINAL,Pittsburgh Panthers,Rider Broncs
8,401918955,Campbell Fighting Camels at NC State Wolfpack,2026-11-03T00:30Z,STATUS_FINAL,NC State Wolfpack,Campbell Fighting Camels
9,401918967,Southern Jaguars at SMU Mustangs,2026-11-03T01:00Z,STATUS_FINAL,SMU Mustangs,Southern Jaguars


In [9]:
# Keep only completed games
df_games_final = df_games_summary[
    df_games_summary["status"] == "STATUS_FINAL"
].copy()

# Check the results
print("Total games collected:", len(df_games_summary))
print("Completed games:", len(df_games_final))
print("Scheduled games excluded:", len(df_games_summary) - len(df_games_final))

display(df_games_final.head())

Total games collected: 110
Completed games: 108
Scheduled games excluded: 2


,game_id,game_name,game_date_utc,status,home_team,away_team
0,401920982,Notre Dame Fighting Irish at Villanova Wildcats,2026-11-01T14:30Z,STATUS_FINAL,Villanova Wildcats,Notre Dame Fighting Irish
1,401906769,Miami Hurricanes at Florida Gators,2026-11-02T23:00Z,STATUS_FINAL,Florida Gators,Miami Hurricanes
2,401909562,Army Black Knights at Duke Blue Devils,2026-11-02T23:30Z,STATUS_FINAL,Duke Blue Devils,Army Black Knights
3,401918947,Chattanooga Mocs at Louisville Cardinals,2026-11-03T02:00Z,STATUS_FINAL,Louisville Cardinals,Chattanooga Mocs
4,401918073,Stetson Hatters at Clemson Tigers,2026-11-03T00:00Z,STATUS_FINAL,Clemson Tigers,Stetson Hatters
